# AgriFM encoder: input contract & forward

AgriFM is a multi-source temporal foundation model: per-sensor 3D patch embeds feed one
shared Video-Swin-Transformer backbone. We use its Sentinel-2 branch as a frozen `(N, 1024)`
encoder. This notebook shows the input contract and runs a real forward pass.

## Input contract: an S2 time series, not a single chip

AgriFM is spatiotemporal (frames x H x W). The benchmark stores pixel/field S2 time series, so the
wrapper: reorders to AgriFM's 10 bands, resamples to 32 frames, normalizes, then expands each
timestep to a constant spatial tile before the encoder pools to one vector.

In [ ]:
# Bootstrap: find repo root and load a model module by path (src has no __init__ files).
import sys, importlib.util
from pathlib import Path
from types import SimpleNamespace
import numpy as np

REPO = Path.cwd()
while not (REPO / 'src').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m; spec.loader.exec_module(m)
    return m

In [ ]:
agrifm = load('agrifm_mod', 'src/models/agrifm.py')
print('embedding dim :', agrifm.AGRIFM_EMBEDDING_DIM)
print('S2 band order :', agrifm.AGRIFM_S2_BANDS)
print('num frames    :', agrifm.AGRIFM_NUM_FRAMES)
print('patch size    :', agrifm.AGRIFM_PATCH_SIZE)
print('repo path     :', agrifm.DEFAULT_REPO_PATH, '(exists:', agrifm.DEFAULT_REPO_PATH.exists(), ')')
print('weights path  :', agrifm.DEFAULT_WEIGHTS_PATH, '(exists:', agrifm.DEFAULT_WEIGHTS_PATH.exists(), ')')

In [ ]:
# A tiny synthetic Benchmark (duck-typed): only the S2 attributes the encoder reads.
# Matches what src/dataio/get_input.py produces, but needs no data on disk.
N, T = 4, 24
rng = np.random.default_rng(0)
bench = SimpleNamespace(
    s2=(rng.random((N, T, 11)) * 6000).astype('float32'),   # B2..B12, NDVI (reflectance-ish)
    s2_mask=np.ones((N, T), 'float32'),
    s2_bands=['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12','NDVI'],
    n_samples=N,
)
print('synthetic bench:', N, 'samples x', T, 'timesteps')

## How one datapoint becomes model input

`to_agrifm_series` reorders to AgriFM's 10 bands, picks 32 evenly-spaced frames, normalizes with
AgriFM's band statistics, and zeros masked frames. Result: `(N, 32, 10)`.

In [ ]:
enc = agrifm.AgriFMEncoder(device='cpu', batch_size=2, tile_size=16)
series = enc.to_agrifm_series(bench)
print('series shape:', series.shape, '(N, 32 frames, 10 bands)')
print('one sample, first 3 frames:')
print(np.round(series[0, :3], 2))

In [ ]:
# Single datapoint: the normalized 10-band series AgriFM sees for field 0.
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4))
for b, name in enumerate(agrifm.AGRIFM_S2_BANDS):
    plt.plot(series[0, :, b], marker='.', label=name)
plt.title('AgriFM input: one field, 10 normalized S2 bands x 32 frames')
plt.xlabel('frame'); plt.ylabel('normalized reflectance'); plt.legend(ncol=5, fontsize=8)
plt.tight_layout(); plt.show()

## Forward pass: embedding (needs ../AgriFM + AgriFM.pth)

Loads the released S2 encoder with mmcv stubbed out (deps: torch, timm, einops, mmengine) and
runs the real Video-Swin forward to `(N, 1024)`. Because the embedding is computed from the input
series, dropping later frames changes it: AgriFM is stress-testable.

In [ ]:
try:
    emb = enc.encode(bench)
    print('embedding:', emb.shape, emb.dtype)
    drop = SimpleNamespace(**{**bench.__dict__})
    drop.s2 = bench.s2.copy(); drop.s2[:, T//2:, :] = 0   # temporal drop
    delta = float(np.abs(emb - enc.encode(drop)).max())
    print('condition-sensitive (temporal drop): max delta', round(delta, 3))
except Exception as e:
    print('Skipped real forward:', type(e).__name__, e)
    print('Get the model: git clone https://github.com/flyakon/AgriFM ../AgriFM')
    print('Get weights   : curl -L https://glass.hku.hk/casual/AgriFM/AgriFM.pth -o ../AgriFM/AgriFM.pth')

## Caveats

: S2-only. AgriFM is multi-sensor; we use the Sentinel-2 branch + shared backbone.
: Constant-tile hack. Pixel/field series have no spatial extent, so each frame is tiled to a
  constant `tile_size`; the temporal axis carries the signal.
: mmcv-free. The official stack wants compiled MMCV (no Python 3.13 wheel); we stub mmseg's
  registries and import the repo's real `SwinTransformer3D`. Weights are CC0 from GLASS.